# Import lib etc

In [1]:
pip install "kagglehub[pandas-datasets]"

Note: you may need to restart the kernel to use updated packages.


Import dataframe & pick only 3 games

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "kieranpoc/steam-reviews", "weighted_score_above_08.csv")

100%|██████████| 299M/299M [00:09<00:00, 34.4MB/s] 

Extracting zip of weighted_score_above_08.csv...



/opt/anaconda3/lib/python3.12/site-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


In [4]:
games = [
    "Counter-Strike 2",
    "PUBG: BATTLEGROUNDS",
    "Grand Theft Auto V"
]

df_3games = df[df["game"].isin(games)].copy()

print(df_3games.shape)
df_3games["game"].value_counts()

(7365, 24)


game
Counter-Strike 2       3017
PUBG: BATTLEGROUNDS    2404
Grand Theft Auto V     1944
Name: count, dtype: int64

# 1. Prepare Gemini Labeling Prompt


In [ ]:
with open("../prompts/codebook_exact.txt", "r") as f:
    SYSTEM = f.read()

GEMINI_MODEL = "gemini-3.5-flash-lite"
TEMPERATURE = 0

SCHEMA = {
    "type": "object",
    "properties": {
        "harm_mention": {"type": "string", "enum": ["yes", "no", "uncertain"]},
        "harm_reason": {"type": "string"},
        "irony": {"type": "string", "enum": ["yes", "no", "uncertain"]},
        "irony_reason": {"type": "string"},
        "gender_age": {"type": "string", "enum": ["none", "gender", "age", "both", "uncertain"]},
        "gender_age_reason": {"type": "string"}
    },
    "required": [
        "harm_mention", "harm_reason",
        "irony", "irony_reason",
        "gender_age", "gender_age_reason"
    ]
}


# 2. Set Up Gemini API and Labeling Function


In [ ]:
pip install google-genai python-dotenv


In [ ]:
import os
import json
import time
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# Adjust automatically to the recommendation column in the source data
RECOMMENDATION_COL = "recommendation" if "recommendation" in df_3games.columns else "voted_up"

def make_payload(row):
    recommendation = row[RECOMMENDATION_COL]
    if isinstance(recommendation, bool):
        recommendation = "Recommended" if recommendation else "Not recommended"

    return {
        "game": row["game"],
        "review": row["review"],
        "recommendation": recommendation
    }

def label_review(row):
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=json.dumps(make_payload(row), ensure_ascii=False),
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM,
            temperature=TEMPERATURE,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )
    return json.loads(response.text)


# 3. Run Batch Scale-Up


In [ ]:
OUTPUT_PATH = "../data/gemini_scaleup_results.csv"
WAIT_SECONDS = 4.2  # keeps requests below the 15 requests/minute pilot limit

# Use Steam recommendation ID when available; otherwise use the dataframe index
work = df_3games.copy()
if "recommendationid" in work.columns:
    work["case_id"] = work["recommendationid"].astype(str)
else:
    work["case_id"] = work.index.astype(str)

# Resume from previous output if the batch was interrupted
if os.path.exists(OUTPUT_PATH):
    existing = pd.read_csv(OUTPUT_PATH, dtype={"case_id": str})
    completed = set(existing["case_id"])
else:
    completed = set()

for i, row in work.iterrows():
    case_id = str(row["case_id"])
    if case_id in completed:
        continue

    for attempt in range(3):
        try:
            result = label_review(row)
            output = {
                "case_id": case_id,
                "game": row["game"],
                "review": row["review"],
                "recommendation": make_payload(row)["recommendation"],
                **result
            }

            pd.DataFrame([output]).to_csv(
                OUTPUT_PATH,
                mode="a",
                header=not os.path.exists(OUTPUT_PATH),
                index=False
            )
            completed.add(case_id)
            print(f"{len(completed)}/{len(work)} completed")
            break

        except Exception as e:
            print(f"Error on {case_id}, attempt {attempt + 1}: {e}")
            if attempt < 2:
                time.sleep(60)

    time.sleep(WAIT_SECONDS)
